# Part 12: XRPC Method Dispatch

`XrpcMethodRegistry`. XRPC is the ATProto RPC protocol — methods are identified by NSIDs
(dot-separated names like `com.atproto.server.createSession`) and routed to handler blocks.

**What you'll learn:**

- How NSIDs map to ObjC method names
- Dispatch table registration and lookup
- Auth validation before dispatch
- Input validation guards

**Estimated Time:** 20-25 minutes

## Step 1: XRPC Dispatcher

The dispatcher stores handler blocks in a dictionary keyed by method NSID.
`XrpcDispatcher.registerMethod:handler:` does the same thing.

In [ ]:
@interface XrpcDispatcher : NSObject
@property (nonatomic, strong) NSMutableDictionary *handlers;
- (void)registerMethod:(NSString *)methodId handler:(id)handler;
- (void)unregisterMethod:(NSString *)methodId;
- (NSDictionary *)dispatchMethod:(NSString *)methodId params:(NSDictionary *)params;
@end

@implementation XrpcDispatcher
- (instancetype)init {
    self = [super init];
    if (self) { _handlers = [NSMutableDictionary dictionary]; }
    return self;
}
- (void)registerMethod:(NSString *)methodId handler:(id)handler {
    [_handlers setObject:handler forKey:methodId];
    NSLog(@"Registered: %@", methodId);
}
- (void)unregisterMethod:(NSString *)methodId {
    [_handlers removeObjectForKey:methodId];
    NSLog(@"Unregistered: %@", methodId);
}
- (NSDictionary *)dispatchMethod:(NSString *)methodId params:(NSDictionary *)params {
    id handler = [_handlers objectForKey:methodId];
    if (handler == nil) {
        return @{@"error": @"MethodNotFound", @"method": methodId};
    }
    NSLog(@"Dispatching %@", methodId);
    // In real code, handler is a block invoked with (request, response)
    // Here we simulate by returning a result dict
    return @{@"status": @"ok", @"method": methodId, @"params": params};
}
@end

XrpcDispatcher *disp = [[XrpcDispatcher alloc] init];
[disp registerMethod:@"com.atproto.server.createSession" handler:@"sessionHandler"];
[disp registerMethod:@"com.atproto.repo.createRecord" handler:@"recordHandler"];

NSDictionary *result = [disp dispatchMethod:@"com.atproto.server.createSession" params:@{@"handle": @"alice.bsky.social"}];
NSLog(@"Result: %@", result);

NSDictionary *missing = [disp dispatchMethod:@"com.atproto.unknown" params:@{}];
NSLog(@"Missing: %@", missing);

## Step 2: NSID Helper

Garazyk uses a naming convention: `registerComAtprotoServerCreateSession:` maps to the NSID
`com.atproto.server.createSession`. The conversion splits on capital letters and inserts dots.

In [ ]:
@interface NSIDHelper : NSObject
- (NSString *)camelCaseToNSID:(NSString *)camelCase;
- (NSString *)nsidToCamelCase:(NSString *)nsid;
@end

@implementation NSIDHelper
- (NSString *)camelCaseToNSID:(NSString *)camelCase {
    // Strip "register" prefix, then convert CamelCase to dot-separated
    NSString *name = camelCase;
    if ([name hasPrefix:@"register"]) {
        name = [name substringFromIndex:8]; // len("register")
    }
    // Insert dot before each capital letter (except the first)
    NSMutableArray *parts = [NSMutableArray array];
    NSString *current = [@"" mutableCopy];
    for (int i = 0; i < [name length]; i++) {
        char c = [name characterAtIndex:i];
        if (c >= 'A' && c <= 'Z') {
            if ([current length] > 0) {
                [parts addObject:current];
                current = [@"" mutableCopy];
            }
            // Lowercase the capital
            current = [current stringByAppendingString:[NSString stringWithFormat:@"%c", c + 32]];
        } else {
            current = [current stringByAppendingString:[NSString stringWithFormat:@"%c", c]];
        }
    }
    if ([current length] > 0) {
        [parts addObject:current];
    }
    // Join with dots
    NSString *result = @"";
    for (int i = 0; i < [parts count]; i++) {
        if (i > 0) result = [result stringByAppendingString:@"."];
        result = [result stringByAppendingString:[parts objectAtIndex:i]];
    }
    return result;
}
- (NSString *)nsidToCamelCase:(NSString *)nsid {
    // Split on dots, capitalize each segment after the first
    NSArray *parts = [nsid componentsSeparatedByString:@"."];
    NSString *result = @"register";
    for (int i = 0; i < [parts count]; i++) {
        NSString *part = [parts objectAtIndex:i];
        if ([part length] == 0) continue;
        // Capitalize first letter
        char first = [part characterAtIndex:0];
        if (first >= 'a' && first <= 'z') first = first - 32;
        NSString *rest = [part substringFromIndex:1];
        result = [result stringByAppendingString:[NSString stringWithFormat:@"%c%@", first, rest]];
    }
    return result;
}
@end

NSIDHelper *nsid = [[NSIDHelper alloc] init];
NSLog(@"%@", [nsid camelCaseToNSID:@"registerComAtprotoServerCreateSession"]);
NSLog(@"%@", [nsid camelCaseToNSID:@"registerAppBskyFeedGetTimeline"]);
NSLog(@"%@", [nsid nsidToCamelCase:@"com.atproto.server.createSession"]);
NSLog(@"%@", [nsid nsidToCamelCase:@"app.bsky.feed.getTimeline"]);

## Step 3: Auth Helper

Before dispatching, the auth helper validates tokens. `XrpcAuthHelper` checks JWT signatures and
scopes. Here we use simplified token strings.

In [ ]:
@interface XrpcAuthHelper : NSObject
- (BOOL)validateToken:(NSString *)token forScope:(NSString *)scope;
- (NSString *)extractDidFromToken:(NSString *)token;
@end

@implementation XrpcAuthHelper
- (BOOL)validateToken:(NSString *)token forScope:(NSString *)scope {
    // Simplified: real code verifies JWT signature and claims
    if (token == nil || [token length] == 0) return NO;
    // Token must contain the scope
    if (scope != nil && ![token containsString:scope]) return NO;
    return YES;
}
- (NSString *)extractDidFromToken:(NSString *)token {
    // Simplified: real code decodes JWT payload
    // Token format: "access-<did>-<scope>"
    NSArray *parts = [token componentsSeparatedByString:@"-"];
    if ([parts count] >= 3) {
        return [parts objectAtIndex:1];
    }
    return nil;
}
@end

XrpcAuthHelper *auth = [[XrpcAuthHelper alloc] init];
NSLog(@"Valid token: %d", [auth validateToken:@"access-did:plc:abc-session" forScope:@"session"]);
NSLog(@"Wrong scope: %d", [auth validateToken:@"access-did:plc:abc-session" forScope:@"admin"]);
NSLog(@"Empty token: %d", [auth validateToken:@"" forScope:@"session"]);
NSLog(@"DID: %@", [auth extractDidFromToken:@"access-did:plc:abc-session"]);

## Step 4: Input Validator

services checks handles, DIDs, and collection names before they reach service logic. Validation
happens at the boundary.

In [ ]:
@interface XrpcInputValidator : NSObject
- (BOOL)validateHandle:(NSString *)handle;
- (BOOL)validateDid:(NSString *)did;
- (BOOL)validateCollection:(NSString *)collection;
@end

@implementation XrpcInputValidator
- (BOOL)validateHandle:(NSString *)handle {
    if (handle == nil || [handle length] < 3) return NO;
    if (![handle containsString:@"."]) return NO;
    return YES;
}
- (BOOL)validateDid:(NSString *)did {
    if (did == nil) return NO;
    if ([did hasPrefix:@"did:plc:"]) return YES;
    if ([did hasPrefix:@"did:web:"]) return YES;
    return NO;
}
- (BOOL)validateCollection:(NSString *)collection {
    // Collection is an NSID: at least two dot-separated parts
    if (collection == nil) return NO;
    if (![collection containsString:@"."]) return NO;
    return YES;
}
@end

XrpcInputValidator *validator = [[XrpcInputValidator alloc] init];
NSLog(@"Handle 'alice.bsky.social': %d", [validator validateHandle:@"alice.bsky.social"]);
NSLog(@"Handle 'nodot': %d", [validator validateHandle:@"nodot"]);
NSLog(@"DID 'did:plc:abc': %d", [validator validateDid:@"did:plc:abc"]);
NSLog(@"DID 'invalid': %d", [validator validateDid:@"invalid"]);
NSLog(@"Collection 'app.bsky.feed.post': %d", [validator validateCollection:@"app.bsky.feed.post"]);
NSLog(@"Collection 'nodot': %d", [validator validateCollection:@"nodot"]);

## Step 5: Full Request Lifecycle

Wire it all together: validate input, check auth, dispatch to handler. This shows the full request
path that `XrpcMethodRegistry` orchestrates.

In [ ]:
// Full request lifecycle: validate -> auth -> dispatch -> response
@interface RequestHandler : NSObject
@property (nonatomic, strong) XrpcDispatcher *dispatcher;
@property (nonatomic, strong) XrpcAuthHelper *authHelper;
@property (nonatomic, strong) XrpcInputValidator *validator;
- (NSDictionary *)handleRequest:(NSString *)methodId
                         params:(NSDictionary *)params
                          token:(NSString *)token;
@end

@implementation RequestHandler
- (instancetype)init {
    self = [super init];
    if (self) {
        _dispatcher = [[XrpcDispatcher alloc] init];
        _authHelper = [[XrpcAuthHelper alloc] init];
        _validator = [[XrpcInputValidator alloc] init];
        [_dispatcher registerMethod:@"com.atproto.server.createSession" handler:@"sessionHandler"];
        [_dispatcher registerMethod:@"com.atproto.repo.createRecord" handler:@"recordHandler"];
    }
    return self;
}
- (NSDictionary *)handleRequest:(NSString *)methodId
                         params:(NSDictionary *)params
                          token:(NSString *)token {
    // Step 1: Validate input
    NSString *handle = [params objectForKey:@"handle"];
    if (handle != nil && ![_validator validateHandle:handle]) {
        return @{@"error": @"InvalidHandle"};
    }
    NSString *did = [params objectForKey:@"did"];
    if (did != nil && ![_validator validateDid:did]) {
        return @{@"error": @"InvalidDid"};
    }
    NSString *collection = [params objectForKey:@"collection"];
    if (collection != nil && ![_validator validateCollection:collection]) {
        return @{@"error": @"InvalidCollection"};
    }
    // Step 2: Check auth
    if (![_authHelper validateToken:token forScope:@"session"]) {
        return @{@"error": @"AuthenticationRequired"};
    }
    // Step 3: Dispatch
    return [_dispatcher dispatchMethod:methodId params:params];
}
@end

RequestHandler *rh = [[RequestHandler alloc] init];

// Valid request
NSDictionary *r1 = [rh handleRequest:@"com.atproto.server.createSession"
    params:@{@"handle": @"alice.bsky.social"}
    token:@"access-did:plc:abc-session"];
NSLog(@"Valid: %@", r1);

// Invalid handle
NSDictionary *r2 = [rh handleRequest:@"com.atproto.server.createSession"
    params:@{@"handle": @"nodot"}
    token:@"access-did:plc:abc-session"];
NSLog(@"Invalid handle: %@", r2);

// No auth
NSDictionary *r3 = [rh handleRequest:@"com.atproto.repo.createRecord"
    params:@{@"collection": @"app.bsky.feed.post"}
    token:@""];
NSLog(@"No auth: %@", r3);

## Exercise

Add middleware support to `XrpcDispatcher`. Implement `registerMethod:middleware:handler:` that
stores a validation block alongside the handler. In `dispatchMethod:params:`, run the middleware
first — if it returns NO, return an error dict instead of dispatching.

Hint: store middleware in a parallel dictionary keyed by method NSID.

In [ ]:
// Exercise: implement registerMethod:middleware:handler:
// Your code here...

## Solution

In [ ]:
// Solution: middleware dictionary checked before dispatch
@interface XrpcDispatcher2 : NSObject
@property (nonatomic, strong) NSMutableDictionary *handlers;
@property (nonatomic, strong) NSMutableDictionary *middlewares;
- (void)registerMethod:(NSString *)methodId middleware:(id)middleware handler:(id)handler;
- (NSDictionary *)dispatchMethod:(NSString *)methodId params:(NSDictionary *)params;
@end

@implementation XrpcDispatcher2
- (instancetype)init {
    self = [super init];
    if (self) {
        _handlers = [NSMutableDictionary dictionary];
        _middlewares = [NSMutableDictionary dictionary];
    }
    return self;
}
- (void)registerMethod:(NSString *)methodId middleware:(id)middleware handler:(id)handler {
    [_handlers setObject:handler forKey:methodId];
    if (middleware != nil) {
        [_middlewares setObject:middleware forKey:methodId];
    }
    NSLog(@"Registered %@ with middleware", methodId);
}
- (NSDictionary *)dispatchMethod:(NSString *)methodId params:(NSDictionary *)params {
    // Check middleware first
    id middleware = [_middlewares objectForKey:methodId];
    if (middleware != nil) {
        // Simplified: real middleware is a block returning BOOL
        // Here we check if params contain required "auth" key
        if ([params objectForKey:@"auth"] == nil) {
            NSLog(@"Middleware rejected: %@", methodId);
            return @{@"error": @"AuthRequired", @"method": methodId};
        }
    }
    id handler = [_handlers objectForKey:methodId];
    if (handler == nil) {
        return @{@"error": @"MethodNotFound"};
    }
    return @{@"status": @"ok", @"method": methodId};
}
@end

// Test it
XrpcDispatcher2 *d2 = [[XrpcDispatcher2 alloc] init];
[d2 registerMethod:@"com.atproto.server.createSession" middleware:@"authCheck" handler:@"sessionHandler"];
[d2 registerMethod:@"com.atproto.server.describeServer" middleware:nil handler:@"describeHandler"];

// With auth
NSDictionary *ok = [d2 dispatchMethod:@"com.atproto.server.createSession" params:@{@"auth": @"yes"}];
NSLog(@"With auth: %@", ok);

// Without auth
NSDictionary *no = [d2 dispatchMethod:@"com.atproto.server.createSession" params:@{}];
NSLog(@"No auth: %@", no);

// No middleware
NSDictionary *desc = [d2 dispatchMethod:@"com.atproto.server.describeServer" params:@{}];
NSLog(@"No middleware: %@", desc);

## Next Steps

You have learned about ATProto's core data structures and protocols. Continue to explore how these
concepts apply when building a PDS.

## What to Read Next

You now understand XRPC dispatch. Next:

- **Part 13: Account Service** — handle validation, invite codes, sessions
- **Part 14: Record Writes** — record CRUD, CID addressing, commits